# Ładowanie czystych danych do PostgreSQL

Ładuje sześć czystych tabel z `data/clean/` do bazy `housing`.

**Przed uruchomieniem:** należy uruchomić raz w bazie [`sql/schema.sql`](../sql/schema.sql) (definiuje typy kolumn, klucze główne i obce oraz reguły poprawności).

Dane są ładowane poleceniem PostgreSQL `COPY` w jednej transakcji: albo ładują się wszystkie tabele, albo nic się nie zmienia. Ponowne uruchomienie notebooka zastępuje dane. Kod ładowania jest w [`load_db.py`](load_db.py).

In [1]:
from getpass import getpass
from pathlib import Path

import pandas as pd

import load_db as db

CLEAN_DIR = Path("../data/clean")

engine = db.get_engine(password=getpass("PostgreSQL password: "))
with engine.connect() as conn:
    print(conn.exec_driver_sql("SELECT version();").scalar())

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


## Ładowanie wszystkich tabel

In [2]:
loaded = db.load_all(engine, CLEAN_DIR)
loaded

,table,rows_in_csv
0,cities,15
1,listings,128608
2,sale_snapshots,192356
3,rent_snapshots,70016
4,nbp_reference_rate,47
5,gus_wages,45


## Kontrolam - liczba wierszy w bazie i w plikach CSV

In [3]:
check = db.row_counts(engine).merge(
    loaded.rename(columns={"table": "table_name"}), on="table_name", how="left"
)
check["match"] = check["rows_in_csv"].isna() | (check["rows_in_csv"] == check["rows_in_database"])
check

,table_name,rows_in_database,rows_in_csv,match
0,months,96,NaN,True
1,cities,15,15.0,True
2,gus_wages,45,45.0,True
3,nbp_reference_rate,47,47.0,True
4,rent_snapshots,70016,70016.0,True
5,sale_snapshots,192356,192356.0,True
6,listings,128608,128608.0,True


## Pierwsze zapytanie do bazy
Mediana ceny ofertowej za m² w czerwcu 2024 według miast – szybkie potwierdzenie, że tabele i złączenia działają.

In [4]:
query = '''
SELECT c.city_name,
       COUNT(*)                                                   AS listings,
       ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY s.price_per_m2)::numeric, 0) AS median_price_per_m2
FROM sale_snapshots s
JOIN listings l ON l.listing_key = s.listing_key
JOIN cities   c ON c.city_key    = l.city_key
WHERE s.snapshot_month = DATE '2024-06-01'
GROUP BY c.city_name
ORDER BY median_price_per_m2 DESC;
'''
pd.read_sql(query, engine)

,city_name,listings,median_price_per_m2
0,Warszawa,6872,17923.0
1,Kraków,3156,16322.0
2,Gdańsk,1916,14100.0
3,Wrocław,2239,13089.0
4,Gdynia,806,12646.0
5,Poznań,785,11019.0
6,Rzeszów,162,10797.0
7,Lublin,568,9891.0
8,Białystok,231,9884.0
9,Szczecin,640,9224.0
